# 05. XAI — SHAP / LIME / 자연어 설명

JD의 XAI 항목을 정면 대응한다.

- **SHAP** — TreeExplainer (LightGBM 파트)
  - Global: summary / beeswarm, mean |SHAP|
  - Local: waterfall, 샘플 단위 기여도
  - Interaction: 변수 쌍 간 상호작용
- **LIME** — 샘플 근방 sparse linear 설명 (SHAP 교차 검증)
- **설명 템플릿 표준화** — 변수 기여도 / 임계 구간 / 상호작용 세 가지 표준 구조
- **자연어 생성** — 현장·경영진이 읽는 한국어 스토리

대응 모듈: `src/career_kia/xai/{shap_utils, lime_utils, explanation_templates, nl_generator}.py`

In [ ]:
import sys, json
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import shap

from career_kia.config import PROJECT_ROOT, PROCESSED_DIR
from career_kia.models.train import load_feature_matrix
from career_kia.xai import shap_utils, lime_utils, explanation_templates, nl_generator

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.grid'] = True

## 1. 모델 로드 및 전역 SHAP

In [ ]:
model = joblib.load(PROJECT_ROOT / 'models_artifacts' / 'hybrid_model.joblib')
X, y, _ = load_feature_matrix()
batch_df = pd.read_parquet(PROCESSED_DIR / 'features.parquet')
proba = model.predict_proba(X)[:, 1]
print(f'샘플 수: {len(X)}, 양성 비율: {y.mean():.3f}')

bundle = shap_utils.explain_batch(model, X, max_samples=2000)
print('SHAP bundle:', bundle.values.shape)

In [ ]:
# 전역 mean |SHAP| 상위 20
top = bundle.top_k(20)
ax = top[::-1].plot.barh(figsize=(8, 7))
ax.set_title('전역 SHAP 상위 20 피처 (mean |SHAP|)')
ax.set_xlabel('mean |SHAP|')
plt.tight_layout()
plt.show()

In [ ]:
# SHAP summary plot (beeswarm) — 상위 10 피처만
top10 = list(bundle.top_k(10).index)
idx10 = [bundle.feature_names.index(f) for f in top10]
shap.summary_plot(bundle.values[:, idx10], bundle.X[top10], show=False)
plt.tight_layout()
plt.show()

## 2. 고위험 배치 — 국소 설명 (SHAP waterfall)

In [ ]:
top_idx = int(np.argmax(proba))
print(f'최고 위험 배치: {batch_df.iloc[top_idx]["batch_id"]} — 예측 고장 확률 {proba[top_idx]*100:.1f}%')

# 단일 샘플 SHAP waterfall
local = shap_utils.explain_batch(model, X.iloc[[top_idx]])
shap.plots.waterfall(shap.Explanation(
    values=local.values[0],
    base_values=local.base_value,
    data=local.X.iloc[0].to_numpy(),
    feature_names=local.feature_names,
), max_display=10, show=False)
plt.tight_layout()
plt.show()

## 3. LIME 으로 교차 검증

In [ ]:
lime_exp = lime_utils.build_lime_explainer(model, X.sample(500, random_state=42), training_labels=y.values[:500])
local_lime = lime_utils.explain_instance(model, X.iloc[top_idx], explainer=lime_exp, num_features=8)
print('LIME 설명 (상위 8 변수):')
for rule, _, weight in local_lime.feature_values:
    print(f'  {weight:+.3f}  {rule}')

## 4. 설명 템플릿 + 자연어 생성

In [ ]:
contribution = explanation_templates.build_contribution(
    local, sample_idx=0,
    prediction=float(proba[top_idx]),
    sample_id=str(batch_df.iloc[top_idx]['batch_id']),
)
thresholds = explanation_templates.infer_thresholds(
    bundle, features=list(bundle.top_k(8).index), bins=10, min_shap=0.05,
)
inter_values = shap_utils.interaction_values(model, X, max_samples=500)
interactions = explanation_templates.build_interactions(inter_values, bundle.feature_names, k=5)

batch_exp = explanation_templates.BatchExplanation(
    contribution=contribution, thresholds=thresholds, interactions=interactions,
)

paragraph = nl_generator.batch_explanation_to_paragraph(batch_exp)
print(paragraph)

## 5. 사전 계산 산출물

`make xai` 는 고위험 상위 10개 배치에 대한 자연어 설명을 `xai_artifacts/top_risk_narratives.txt` 로 저장한다. 대시보드와 리포트에서 그대로 재사용된다.

In [ ]:
narr_path = PROJECT_ROOT / 'xai_artifacts' / 'top_risk_narratives.json'
if narr_path.exists():
    narratives = json.loads(narr_path.read_text(encoding='utf-8'))
    print(f'저장된 설명문 수: {len(narratives)}')
    print('\n첫 번째 샘플:')
    print(narratives[0]['narrative'][:500])

## 요약

- 전역 SHAP → 피처 중요도 + 방향성
- 국소 SHAP + LIME 이 동일한 기여 구조를 보이면 설명 신뢰도 ↑
- 표준 템플릿(기여·임계·상호작용) + 자연어 생성으로 비전문가 전달
- 산출물은 대시보드 2페이지('불량원인 설명')와 3페이지('변수중요도 트렌드')에 연결

→ 다음 노트북(`06_인과추론_DAG_개입효과.ipynb`)에서 "어떤 공정 조건을 바꾸면 불량률이 얼마나 줄어드는가" 에 답한다.